# 04 · Embed and store

Stage 4 of the indexing pipeline (see `ARCHITECTURE.md`).

Turns every chunk from `03_chunk` into a vector with `all-MiniLM-L6-v2` and saves them in a Chroma index on disk, **one collection per strategy** (`acre_fixed`, `acre_semantic`), so retrieval can be compared later.

- Vectors are normalised and the collections use **cosine** distance (lower = closer).
- Re-running is safe: each collection is deleted and rebuilt from the JSONL files.

⚠️ Memory: this notebook peaks at ~700 MB RAM (mostly the embedding model, already in the HF cache) and takes 1–2 minutes on CPU. Close other notebook kernels first if RAM is tight.

Output: `data/chroma_db/`.

In [1]:
import json
import os
from pathlib import Path

# Hugging Face warns that Windows can't use symlinks in its model cache; caching still works.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

CHUNKS_DIR = Path("../data/chunks")
CHROMA_DIR = Path("../data/chroma_db")
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
STRATEGIES = ["fixed", "semantic"]


def load_chunks(name):
    lines = (CHUNKS_DIR / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()
    records = [json.loads(line) for line in lines if line]
    return [Document(id=r["id"], page_content=r["text"], metadata=r["metadata"]) for r in records]


chunks = {name: load_chunks(name) for name in STRATEGIES}
for name, docs in chunks.items():
    print(f"{name}: {len(docs)} chunks")

fixed: 324 chunks
semantic: 330 chunks


## Embed and store

Loads the model once, then builds each collection. Chunks longer than 256 tokens are truncated for the vector only; the full text is stored and will still go to Claude.

In [2]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
)

stores = {}
for name, docs in chunks.items():
    store = Chroma(
        collection_name=f"acre_{name}",
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR),
        collection_metadata={"hnsw:space": "cosine"},
    )
    store.reset_collection()  # start clean so re-runs don't duplicate or keep stale chunks
    store.add_documents(docs, ids=[d.id for d in docs])
    stores[name] = store
    print(f"acre_{name}: {store._collection.count()} vectors stored")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

acre_fixed: 324 vectors stored
acre_semantic: 330 vectors stored


## Sanity check

Asks both collections the same question and shows the top 5 hits. This is plain similarity search, not the real retrieve/rerank step yet; it only confirms the index works and points to sensible places. Change `QUESTION` to try others.

In [3]:
QUESTION = "What was the net interest margin this quarter?"

rows = []
for name, store in stores.items():
    for rank, (doc, distance) in enumerate(store.similarity_search_with_score(QUESTION, k=5), start=1):
        rows.append({
            "strategy": name,
            "rank": rank,
            "distance": round(distance, 3),
            "chunk_id": doc.metadata["chunk_id"],
            "type": doc.metadata["type"],
            "section": doc.metadata["section"],
            "preview": doc.page_content[:120].replace("\n", " "),
        })

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows)

,strategy,rank,distance,chunk_id,type,section,preview
0,fixed,1,0.364,fixed-0280,table,Part I · Item 1. Consolidated Financial Statements,Table: Net Interest Margin and Interest Expense / Net interest margin in the Company’s consolidated statements of op...
1,fixed,2,0.385,fixed-0310,table,Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,Table: The following tables set forth select details of our consolidated results of operations for the three and six...
2,fixed,3,0.389,fixed-0180,text,Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,"For the three months ended June 30, 2026 and 2025, net interest margin was approximately $8.6 million and $7.0 milli..."
3,fixed,4,0.398,fixed-0179,text,Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,RESULTS OF OPERATIONS The following table sets forth a summary of our consolidated results of operations for the th...
4,fixed,5,0.406,fixed-0182,text,Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,"For the six months ended June 30, 2026 and 2025, net interest margin was approximately $16.1 million and $16.3 milli..."
5,semantic,1,0.364,semantic-0286,table,Part I · Item 1. Consolidated Financial Statements,Table: Net Interest Margin and Interest Expense / Net interest margin in the Company’s consolidated statements of op...
6,semantic,2,0.374,semantic-0045,text,Part I · Item 1. Consolidated Financial Statements,Net interest margin in the Company’s consolidated statements of operations serves to measure the performance of the ...
7,semantic,3,0.385,semantic-0316,table,Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,Table: The following tables set forth select details of our consolidated results of operations for the three and six...
8,semantic,4,0.389,semantic-0185,text,Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,"For the three months ended June 30, 2026 and 2025, net interest margin was approximately $8.6 million and $7.0 milli..."
9,semantic,5,0.402,semantic-0184,text,Part I · Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations,Management’s Discussion and Analysis of Financial Condition and Results of Operations” in our 2025 Annual Report on ...


## Reopen from disk

Later stages will load the index like this instead of re-embedding. Checks the saved collections can be opened fresh.

In [4]:
for name in STRATEGIES:
    reopened = Chroma(
        collection_name=f"acre_{name}",
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR),
    )
    print(f"acre_{name}: {reopened._collection.count()} vectors on disk")

size_mb = sum(p.stat().st_size for p in CHROMA_DIR.rglob("*") if p.is_file()) / 1_000_000
print(f"Index size: {size_mb:.1f} MB at {CHROMA_DIR}")

acre_fixed: 324 vectors on disk
acre_semantic: 330 vectors on disk
Index size: 7.4 MB at ..\data\chroma_db
